<a href="https://colab.research.google.com/github/rakshitshah280701/InstructAware/blob/main/Option2_Training_T5_with_Final_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# With T5-small

In [ ]:
!pip install transformers datasets
!pip install --upgrade transformers sentencepiece

!pip install fsspec==2024.10.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import pandas as pd
import ast
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import torch
from collections import Counter


In [ ]:
import pandas as pd
import json
import re  # Import regex for text cleaning

# Paths to the four CSV files
csv_files = [
    "/content/drive/MyDrive/InstructAware/Data/NarrativesData/New/narrative_1.csv",
    "/content/drive/MyDrive/InstructAware/Data/NarrativesData/New/narrative_2.csv",
    "/content/drive/MyDrive/InstructAware/Data/NarrativesData/New/narrative_3.csv",
    "/content/drive/MyDrive/InstructAware/Data/NarrativesData/New/narrative_4.csv"
]

final_data = []

# Function to normalize bounding boxes
def normalize_bbox(bbox, image_width=2880, image_height=1800):
    try:
        bbox = json.loads(bbox) if isinstance(bbox, str) else bbox  # Convert JSON string to list
        x_min, y_min, x_max, y_max = bbox
        return [
            x_min / image_width,
            y_min / image_height,
            (x_max - x_min) / image_width,
            (y_max - y_min) / image_height,
        ]
    except:
        return [0, 0, 0, 0]  # Default for errors

# Function to clean output text
def clean_output_text(text):
    if not isinstance(text, str):  # Check if text is None or not a string
        return ""  # Return an empty string if text is missing

    # Remove numeric prefixes like "1.", "2.", etc.
    text = re.sub(r"^\d+\.\s*", "", text)

    # Remove descriptive headings like "### Detailed Description"
    text = re.sub(r"###\s*(Detailed Description|Concise Description|Simplified Description|Conversational Tone)\s*", "", text, flags=re.IGNORECASE)

    return text.strip()

# Process each file separately
for file_path in csv_files:
    df = pd.read_csv(file_path)

    # Normalize bounding boxes
    df["Bounding Box"] = df["Bounding Box"].apply(normalize_bbox)

    # Group by 'Original Filename' and aggregate bounding boxes and OCR texts
    grouped_df = df.groupby("Original Filename").agg({
        "OCR TEXT": list,  # Collect OCR texts as a list
        "Bounding Box": list,  # Collect Bounding Boxes as a list
        "Narrative": "first"  # Keep the first narrative for each group
    }).reset_index()

    # Convert grouped data into structured input-output format
    for _, row in grouped_df.iterrows():
        input_text = " ".join([
            f"'{ocr}' at {bbox}" for ocr, bbox in zip(row["OCR TEXT"], row["Bounding Box"])
        ])

        final_data.append({
            "Input": input_text,
            "Output": clean_output_text(row["Narrative"])  # Apply cleaning function safely
        })

# Convert to DataFrame
output_df = pd.DataFrame(final_data)

# Save as CSV
csv_output_path = "/content/structured_dataset_for_T5.csv"
output_df.to_csv(csv_output_path, index=False)

print(f"✅ CSV file saved at: {csv_output_path}")

In [ ]:
import pandas as pd

# Load the CSV file
csv_path = "/content/structured_dataset_for_T5.csv"  # Update the path if needed
df = pd.read_csv(csv_path)

# Display first few rows
df.head()

from google.colab import data_table

# Display CSV as an interactive table
data_table.DataTable(df)

In [ ]:
# Step 6: Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(output_df[['Input', 'Output']])
hf_dataset = hf_dataset.train_test_split(test_size=0.2)

hf_dataset


In [ ]:
# Tokenization
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def preprocess_function(examples):
    inputs = tokenizer(
        examples['Input'], max_length=512, truncation=True, padding="max_length"
    )
    outputs = tokenizer(
        examples['Output'], max_length=128, truncation=True, padding="max_length"
    )
    inputs['labels'] = outputs['input_ids']
    return inputs

# Apply tokenization
tokenized_dataset = hf_dataset.map(preprocess_function, batched=True)


In [ ]:
# Fine-Tuning T5
model = T5ForConditionalGeneration.from_pretrained("t5-small")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-finetuned",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=30,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to="tensorboard",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train the model
trainer.train()

In [ ]:
# import pandas as pd
# import torch

# # Move model to device
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device)

# # Predict for test dataset
# predictions = []
# all_inputs = hf_dataset["test"]["Input"]

# for input_text in all_inputs:
#     tokenized_input = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
#     tokenized_input = {key: value.to(device) for key, value in tokenized_input.items()}

#     outputs = model.generate(
#         tokenized_input["input_ids"],
#         max_length=256,
#         num_beams=10,
#         early_stopping=True
#     )

#     predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     predictions.append({"input_text": input_text, "predicted_narrative": predicted_text})

# # Save Predictions to CSV
# predictions_df = pd.DataFrame(predictions)
# predictions_df.to_csv("predicted_narratives_full_t5_small.csv", index=False)

# print(f"✅ Predictions for full dataset saved to predicted_narratives_full_t5_small.csv")


import pandas as pd
import torch

# Move model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Predict for test dataset
predictions = []
all_inputs = hf_dataset["test"]["Input"]
all_original_outputs = hf_dataset["test"]["Output"]  # Get the original target outputs

for input_text, original_output in zip(all_inputs, all_original_outputs):
    tokenized_input = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    tokenized_input = {key: value.to(device) for key, value in tokenized_input.items()}

    outputs = model.generate(
        tokenized_input["input_ids"],
        max_length=256,
        num_beams=10,
        early_stopping=True
    )

    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Store input, predicted, and original output
    predictions.append({
        "input_text": input_text,
        "original_output": original_output,  # Add the original output
        "predicted_narrative": predicted_text
    })

# Save Predictions to CSV
predictions_df = pd.DataFrame(predictions)
predictions_df.to_csv("Predicted_Narrative_T5model.csv", index=False)

print(f"✅ Predictions saved to Predicted_Narratives_T5model.csv")


In [ ]:
import pandas as pd

# Load the CSV file
csv_path = "/content/Predicted_Narrative_T5model.csv"  # Update the path if needed
df = pd.read_csv(csv_path)

# Display first few rows
df.head()

from google.colab import data_table

# Display CSV as an interactive table
data_table.DataTable(df)

In [ ]:
save_path = "/content/drive/MyDrive/InstructAware/Code/Option2TransformerT5/New_Result_RunOn_March21/t-5_Finetuned_Weights"

# Save model and tokenizer to Google Drive
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to Google Drive at {save_path}")


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./logs


In [ ]:
!pip install tensorboard


In [ ]:
import shutil

# Define the path to save in Drive
logs_drive_path = "/content/drive/MyDrive/InstructAware/Code/Option2TransformerT5/New_Result_RunOn_March21/logs"

# Copy the logs to Google Drive
shutil.copytree("./logs", logs_drive_path)

print(f"✅ TensorBoard logs saved to Google Drive at {logs_drive_path}")


In [ ]:
import shutil

# Define the path to save in Drive
logs_drive_path = "/content/drive/MyDrive/InstructAware/Code/Option2TransformerT5/New_Result_RunOn_March21/checkpoints"

# Copy the logs to Google Drive
shutil.copytree("/content/t5-finetuned", logs_drive_path)

print(f"✅ All the checkpoints saved to Google Drive at {logs_drive_path}")
